# MR3b — Hourly Final Join
**Inputs**: `mr3/hourly_crash_mr.csv` + `mr3/address_store_mr.csv` + `mr3/type_count_store_mr.csv`
**Output**: `mr3/mr_hourly_final.csv`

Join type: **inner** (zones with both crashes AND stores)

In [5]:
import json
import pandas as pd
from collections import defaultdict

CRASH_INPUT  = 'hourly_crash_mr.csv'
ADDR_INPUT   = 'address_store_mr.csv'
TYPE_INPUT   = 'type_count_store_mr.csv'
OUTPUT       = 'mr_hourly_final.csv'

crashes = pd.read_csv(CRASH_INPUT)
addr    = pd.read_csv(ADDR_INPUT)
types   = pd.read_csv(TYPE_INPUT)

print(f'Hourly crash rows: {len(crashes):,}  |  zones: {crashes["zone_id"].nunique():,}')
print(f'Store zones      : {len(addr):,}')
print(f'Type rows        : {len(types):,}')

Hourly crash rows: 2,708  |  zones: 1,490
Store zones      : 1,823
Type rows        : 6,576


In [6]:
# ── MAP ───────────────────────────────────────────────────────────────────────

addr_lookup = addr.set_index('zone_id').to_dict('index')

type_lookup = defaultdict(dict)
for _, row in types.iterrows():
    type_lookup[row['zone_id']][row['method_of_operation']] = int(row['count'])

def mapper(crash_row):
    zid = crash_row['zone_id']
    if zid not in addr_lookup:
        return None
    store = addr_lookup[zid]
    return {
        'zone_id':           zid,
        'hour':              int(crash_row['hour']),
        'avg_crashes':       crash_row['avg_crashes'],
        'avg_injured':       crash_row['avg_injured'],
        'avg_killed':        crash_row['avg_killed'],
        'store_count':       store['store_count'],
        'active_licenses':   store['active_licenses'],
        'outdated_licenses': store['outdated_licenses'],
        'type_counts_json':  json.dumps(type_lookup.get(zid, {})),
    }

mapped = [mapper(row) for _, row in crashes.iterrows()]
mapped = [v for v in mapped if v is not None]
print(f'Mapped (inner join) rows : {len(mapped):,}')
print(f'Dropped crash-only zones : {len(crashes) - len(mapped):,}')

Mapped (inner join) rows : 1,973
Dropped crash-only zones : 735


In [7]:
# ── REDUCE ────────────────────────────────────────────────────────────────────

out = pd.DataFrame(mapped).sort_values(['zone_id','hour']).reset_index(drop=True)
out.to_csv(OUTPUT, index=False)

print(f'Output rows : {len(out):,}')
print(f'Unique zones: {out["zone_id"].nunique():,}')
print(f'Hours covered: {sorted(out["hour"].unique())}')
print(f'Saved → {OUTPUT}')

sample = out.iloc[0]
print(f'\nSample zone : {sample["zone_id"]}  hour={sample["hour"]}')
print(f'Type counts : {sample["type_counts_json"]}')
out.head()

Output rows : 1,973
Unique zones: 1,024
Hours covered: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23)]
Saved → mr_hourly_final.csv

Sample zone : 8a2a1008804ffff  hour=6
Type counts : {"Tavern Serving Beer Cider Wine And Liquor": 1, "Restaurant Serving Beer, Wine, Liquor, & Cider": 1, "Restaurant Serving Beer Cider Wine And Liquor": 1}


,zone_id,hour,avg_crashes,avg_injured,avg_killed,store_count,active_licenses,outdated_licenses,type_counts_json
0,8a2a1008804ffff,6,1.0,0.0,0.0,3,0,3,"{""Tavern Serving Beer Cider Wine And Liquor"": ..."
1,8a2a1008804ffff,21,1.0,1.0,0.0,3,0,3,"{""Tavern Serving Beer Cider Wine And Liquor"": ..."
2,8a2a1008814ffff,4,1.0,1.0,0.0,6,1,5,"{""Restaurant Serving Beer, Cider And Wine"": 1,..."
3,8a2a1008814ffff,9,1.0,0.0,0.0,6,1,5,"{""Restaurant Serving Beer, Cider And Wine"": 1,..."
4,8a2a1008814ffff,17,1.0,0.0,0.0,6,1,5,"{""Restaurant Serving Beer, Cider And Wine"": 1,..."
